In [35]:
# ============================================================
# 0) DEPENDENCIAS (opcional)
# ============================================================
# !pip install -q openpyxl requests pandas
print("OK")

OK


In [36]:
# ============================================================
# 1) PARAMETROS
# ============================================================
ANO_INICIAL = 2025
ANO_FINAL   = 2025
FLUXO       = "export"   # "export" ou "import"

SH4S = {
    "1301": "Goma Resina",
    "3805": "Terebintina",
    "3806": "Breu e Derivados",
}

print(f"{ANO_INICIAL}-{ANO_FINAL} | {FLUXO} | {list(SH4S.keys())}")

2025-2025 | export | ['1301', '3805', '3806']


In [37]:
# ============================================================
# 2) MAPA MUNICIPIO -> FABRICANTE POR PRODUTO (SH4)
# ============================================================
import unicodedata
import re

def norm_mun(s):
    s = (s or "")

    # troca NBSP (espaço invisível) por espaço normal
    s = s.replace("\u00A0", " ")

    # padroniza hífens/dashes
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")

    # remove acentos
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    # upper + trim
    s = s.strip().upper()

    # padroniza espaços ao redor do hífen: "X- SP" / "X -SP" / "X - SP" => "X - SP"
    s = re.sub(r"\s*-\s*", " - ", s)

    # colapsa múltiplos espaços
    s = " ".join(s.split())

    return s

FABRICANTES = {
    norm_mun("Apiaí - SP"): {"3806":"AS RESINAS","3805":"AS RESINAS","1301":"OUTROS"},
    norm_mun("Avaré - SP"): {"3806":"RB / RESIPIM","3805":"RB / RESIPIM","1301":"RB / RESIPIM"},
    norm_mun("Balneário Pinhal - RS"): {"3806":"NOHVA PINE (IRANI)","3805":"NOHVA PINE (IRANI)","1301":"OUTROS"},
    norm_mun("Barra do Chapéu - SP"): {"3806":"AS RESINAS","3805":"AS RESINAS","1301":"OUTROS"},
    norm_mun("Buri - SP"): {"3806":"PINUS BRASIL","3805":"PINUS BRASIL","1301":"RESINAS BRASIL"},
    norm_mun("Caçador - SC"): {"3806":"IRN RESINAS","3805":"IRN RESINAS","1301":"OUTROS"},
    norm_mun("Campina do Monte Alegre - SP"): {"3806":"RESINEVES","3805":"RESINEVES","1301":"RESINEVES"},
    norm_mun("Campo Largo - PR"): {"3806":"FLORPINUS","3805":"FLORPINUS","1301":"OUTROS"},
    norm_mun("Canguçu - RS"): {"3806":"OUTROS","3805":"OUTROS","1301":"RESINAS BRASIL"},
    norm_mun("Capivari do Sul - RS"): {"3806":"RESINAS JARDIM","3805":"RESINAS JARDIM","1301":"RESINAS JARDIM"},
    norm_mun("Cascavel - PR"): {"3806":"ENOVA","3805":"ENOVA","1301":"AMBAR"},
    norm_mun("Colombo - PR"): {"3806":"OUTROS","3805":"OUTROS","1301":"DEPIMIEL"},
    norm_mun("Itapetininga - SP"): {"3806":"RESINAS BRASIL","3805":"RESINAS BRASIL","1301":"PRIMUS"},
    norm_mun("Itapeva - SP"): {"3806":"RESINEVES","3805":"RESINEVES","1301":"FIRMINO"},
    norm_mun("Itararé - SP"): {"3806":"OUTROS","3805":"OUTROS","1301":"KLOCK"},
    norm_mun("Laranjeiras do Sul - PR"): {"3806":"OUTROS","3805":"OUTROS","1301":"POLESI"},
    norm_mun("Manduri - SP"): {"3806":"RESINAS BRASIL","3805":"RESINAS BRASIL","1301":"RESINAS BRASIL"},
    norm_mun("Mogi Guaçu - SP"): {"3806":"RESINAS BRASIL","3805":"RESINAS BRASIL","1301":"RESINAS BRASIL"},
    norm_mun("Monte Mor - SP"): {"3806":"OCQ","3805":"OCQ","1301":"OCQ"},
    norm_mun("Mostardas - RS"): {"3806":"OUTROS","3805":"OUTROS","1301":"SHENG (OCQ)"},
    norm_mun("Pindamonhangaba - SP"): {"3806":"MAYSTAR","3805":"MAYSTAR","1301":"OUTROS"},
    norm_mun("Ponta Grossa - PR"): {"3806":"HARIMA","3805":"HARIMA","1301":"OUTROS"},
    norm_mun("Rio Grande - RS"): {"3806":"RB SUL","3805":"RB SUL","1301":"AMBAR"},
    norm_mun("Rio Negro - PR"): {"3806":"MENEGHEL","3805":"MENEGHEL","1301":"OUTROS"},
    norm_mun("Salto - SP"): {"3806":"RESINAS BRASIL","3805":"RESINAS BRASIL","1301":"RESINAS BRASIL"},
    norm_mun("São José do Norte - RS"): {"3806":"AMBAR","3805":"AMBAR","1301":"AMBAR"},
    norm_mun("Sengés - PR"): {"3806":"RESINAS BRASIL","3805":"RESINAS BRASIL","1301":"OUTROS"},
    norm_mun("Tavares - RS"): {"3806":"RESINAS JARDIM","3805":"RESINAS JARDIM","1301":"RESINAS JARDIM"},
    norm_mun("Tibagi - PR"): {"3806":"22 DE DEZEMBRO","3805":"22 DE DEZEMBRO","1301":"OUTROS"},
    norm_mun("Tramandaí - RS"): {"3806":"OUTROS","3805":"OUTROS","1301":"JULIO ANTUNES"},
    norm_mun("Guarulhos - SP"): {"3806":"OCQ","3805":"OCQ","1301":"OCQ"},
}

def fabricante_por_mun_prod(municipio, sh4_code):
    k = norm_mun(municipio)
    # Normaliza: int 3806 ou str '3806' -> '3806'
    sh4_str = str(int(sh4_code)) if str(sh4_code).strip().isdigit() else str(sh4_code).strip()
    return FABRICANTES.get(k, {}).get(sh4_str, "OUTROS")

In [38]:
print(norm_mun("Apiaí - SP"), "=>", fabricante_por_mun_prod("Apiaí - SP", "3805"))
print(norm_mun("Barra do Chapéu - SP"), "=>", fabricante_por_mun_prod("Barra do Chapéu - SP", "3806"))
print(norm_mun("Rio Grande - RS"), "=>", fabricante_por_mun_prod("Rio Grande - RS", "3805"))

APIAI - SP => AS RESINAS
BARRA DO CHAPEU - SP => RESINAS BRASIL
RIO GRANDE - RS => RB SUL


In [39]:
# ============================================================
# 3) FUNCOES API + NORMALIZACAO (COM FABRICANTE)
# ============================================================
import requests, time, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

API = "https://api-comexstat.mdic.gov.br/cities"
MAX_RETRIES = 5
BACKOFF_BASE = 15

def buscar(sh4, ano_ini, ano_fim, fluxo="export"):
    payload = {
        "flow": fluxo,
        "monthDetail": True,
        "period": {"from": f"{ano_ini}-01", "to": f"{ano_fim}-12"},
        "filters": [{"filter": "heading", "values": [int(sh4)]}],
        "details": ["city", "state", "country"],
        "metrics": ["metricFOB", "metricKG"],
    }
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    for tentativa in range(1, MAX_RETRIES + 1):
        r = requests.post(API, json=payload, headers=headers, timeout=60, verify=False)
        if r.status_code == 429:
            espera = BACKOFF_BASE * (2 ** (tentativa - 1))
            print(f"\n      ⏳ 429 — tentativa {tentativa}/{MAX_RETRIES}, aguardando {espera}s...", end=" ", flush=True)
            time.sleep(espera)
            continue
        if r.status_code != 200:
            raise requests.HTTPError(r.text, response=r)
        d = r.json()
        return d.get("data", {}).get("list", []) or d.get("list", [])
    raise requests.HTTPError(f"Rate limit persistente apos {MAX_RETRIES} tentativas", response=r)

def normalizar(reg, sh4_code):
    def f(v):
        try: return float(str(v or 0).replace(",", "."))
        except: return 0.0
    ano = reg.get("year", reg.get("coAno", ""))
    mes = reg.get("monthNumber", reg.get("month", reg.get("coMes", "")))
    municipio = reg.get("noMunMinsgUf", reg.get("city", reg.get("noMunmin", "N/D")))
    uf = reg.get("state", reg.get("sgUfMun", "N/D"))
    pais = reg.get("country", "N/D")
    fob = f(reg.get("metricFOB", reg.get("vlFob", 0)))
    kg  = f(reg.get("metricKG", reg.get("kgLiquido", 0)))
    fabricante = fabricante_por_mun_prod(municipio, sh4_code)
    return {
        "ano": ano, "mes": mes, "municipio": municipio, "uf": uf,
        "pais": pais, "sh4": sh4_code, "fabricante": fabricante,
        "fob_usd": fob, "kg_liquido": kg,
        "preco_usd_kg": (fob / kg) if kg > 0 else 0.0,
    }

print("Funcoes OK (retry + pais)")


Funcoes OK


In [42]:
# ============================================================
# 4) TESTE RAPIDO (opcional)
# ============================================================
import pprint

r = requests.post(
    API,
    json={
        "flow": "export",
        "monthDetail": True,
        "period": {"from": "2025-01", "to": "2025-01"},
        "filters": [{"filter": "heading", "values": [1301]}],
        "details": ["city", "state"],
        "metrics": ["metricFOB", "metricKG"],
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    timeout=60,
    verify=False,
)

print(f"HTTP {r.status_code}")
d = r.json()
lista = d.get("data", {}).get("list", []) or d.get("list", [])
print(f"{len(lista)} registros")
pprint.pprint(lista[0] if lista else d)

HTTP 200
16 registros
{'metricFOB': '740955',
 'metricKG': '817950',
 'monthNumber': '01',
 'noMunMinsgUf': 'Rio Grande - RS',
 'state': 'Rio Grande do Sul',
 'year': '2025'}


In [44]:
# ============================================================
# 5) COLETA (TODOS SH4)
# ============================================================
import pandas as pd
todos_dados = {}
erros = []
INTERVALO_ENTRE_SH4 = 20
for idx, (sh4, desc) in enumerate(SH4S.items()):
    if idx > 0:
        print(f"   💤 Aguardando {INTERVALO_ENTRE_SH4}s...")
        time.sleep(INTERVALO_ENTRE_SH4)
    print(f"\n[{sh4}] {desc}...", end=" ", flush=True)
    try:
        brutos = buscar(sh4, ANO_INICIAL, ANO_FINAL, FLUXO)
        todos_dados[sh4] = [normalizar(r, sh4) for r in brutos]
        print(f"{len(todos_dados[sh4])} registros")
    except requests.HTTPError as e:
        code = getattr(e.response, "status_code", None)
        print(f"❌ HTTP {code}")
        try: print("   Detalhe:", e.response.text[:300])
        except: pass
        erros.append(sh4)
        todos_dados[sh4] = []
    except Exception as e:
        print(f"❌ {e}")
        erros.append(sh4)
        todos_dados[sh4] = []
print(f"\nTotal: {sum(len(v) for v in todos_dados.values())} reg")
if erros: print(f"Erros: {erros}")



[1301] Goma Resina... 155 registros

[3805] Terebintina... 208 registros

[3806] Breu e Derivados... 321 registros

Total: 684 reg


In [45]:
# ============================================================
# 6) RESUMOS NA TELA (opcional)
# ============================================================
from IPython.display import display
pd.set_option("display.float_format", "{:,.2f}".format)

for sh4, desc in SH4S.items():
    dados = todos_dados.get(sh4, [])
    if not dados:
        print(f"\n[{sh4}] sem dados")
        continue

    df = pd.DataFrame(dados)
    print(f'\n{"="*55}\n  {desc} — SH4 {sh4} | {len(df)} registros\n{"="*55}')

    print("\nPor UF:")
    r = (
        df.groupby("uf")
        .agg(fob=("fob_usd", "sum"), kg=("kg_liquido", "sum"), muns=("municipio", "nunique"))
        .sort_values("fob", ascending=False)
        .head(10)
    )
    r["usd_kg"] = r["fob"] / r["kg"].replace(0, float("nan"))
    display(r)

    print("\nTop 10 municipios:")
    t = (
        df.groupby(["municipio", "uf", "fabricante"])
        .agg(fob=("fob_usd", "sum"), kg=("kg_liquido", "sum"))
        .sort_values("fob", ascending=False)
        .head(10)
    )
    t["usd_kg"] = t["fob"] / t["kg"].replace(0, float("nan"))
    display(t)



  Goma Resina — SH4 1301 | 155 registros

Por UF:


,fob,kg,muns,usd_kg
uf,,,,
Rio Grande do Sul,"16,434,510.00","16,749,018.00",6,0.98
São Paulo,"12,417,079.00","10,262,080.00",15,1.21
Paraná,"761,862.00","191,800.00",4,3.97
Minas Gerais,"78,803.00","70,000.00",1,1.13
Pará,"74,520.00","3,175.00",3,23.47
Amazonas,"42,041.00","1,711.00",2,24.57
Santa Catarina,"12,567.00","7,710.00",1,1.63
Mato Grosso do Sul,"3,698.00",425.00,1,8.70
Rondônia,"2,379.00",77.00,1,30.90



Top 10 municipios:


,,,fob,kg,usd_kg
municipio,uf,fabricante,,,
São José do Norte - RS,Rio Grande do Sul,OUTROS,"7,847,902.00","8,061,650.00",0.97
Rio Grande - RS,Rio Grande do Sul,OUTROS,"6,743,834.00","6,818,100.00",0.99
Itapeva - SP,São Paulo,OUTROS,"6,359,956.00","5,155,882.00",1.23
Guarulhos - SP,São Paulo,OUTROS,"2,854,963.00","2,811,570.00",1.02
Manduri - SP,São Paulo,OUTROS,"1,203,566.00","1,142,510.00",1.05
Monte Mor - SP,São Paulo,OUTROS,"1,059,480.00","1,080,000.00",0.98
Tramandaí - RS,Rio Grande do Sul,OUTROS,"775,164.00","830,607.00",0.93
Canguçu - RS,Rio Grande do Sul,OUTROS,"579,387.00","538,380.00",1.08
Campinas - SP,São Paulo,OUTROS,"552,898.00","23,583.00",23.44



  Terebintina — SH4 3805 | 208 registros

Por UF:


...

,,,fob,kg,usd_kg
municipio,uf,fabricante,,,
Buri - SP,São Paulo,OUTROS,"14,762,262.00","5,303,100.00",2.78
Salto - SP,São Paulo,OUTROS,"13,973,918.00","4,924,400.00",2.84
Campina do Monte Alegre - SP,São Paulo,OUTROS,"7,312,749.00","2,521,409.00",2.90
Apiaí - SP,São Paulo,OUTROS,"6,866,275.00","2,487,150.00",2.76
Rio Grande - RS,Rio Grande do Sul,OUTROS,"4,482,424.00","1,578,000.00",2.84
Balneário Pinhal - RS,Rio Grande do Sul,OUTROS,"4,434,960.00","1,599,590.00",2.77
Manduri - SP,São Paulo,OUTROS,"3,723,338.00","1,380,000.00",2.70
Monte Mor - SP,São Paulo,OUTROS,"3,553,577.00","1,378,844.00",2.58
Avaré - SP,São Paulo,OUTROS,"3,457,155.00","1,240,000.00",2.79



  Breu e Derivados — SH4 3806 | 321 registros

Por UF:


,fob,kg,muns,usd_kg
uf,,,,
São Paulo,"90,766,203.00","73,597,828.00",27,1.23
Paraná,"22,561,122.00","15,492,405.00",7,1.46
Rio Grande do Sul,"21,602,614.00","20,898,314.00",10,1.03
Rio de Janeiro,"897,870.00","711,414.00",3,1.26
Santa Catarina,"315,886.00","318,481.00",3,0.99
Minas Gerais,"102,934.00","101,000.00",1,1.02



Top 10 municipios:


,,,fob,kg,usd_kg
municipio,uf,fabricante,,,
Buri - SP,São Paulo,OUTROS,"25,449,085.00","20,287,736.00",1.25
Manduri - SP,São Paulo,OUTROS,"17,661,077.00","15,602,000.00",1.13
Campina do Monte Alegre - SP,São Paulo,OUTROS,"11,302,829.00","10,802,600.00",1.05
Sengés - PR,Paraná,OUTROS,"9,616,351.00","8,920,000.00",1.08
Rio Grande - RS,Rio Grande do Sul,OUTROS,"9,573,612.00","8,951,049.00",1.07
Balneário Pinhal - RS,Rio Grande do Sul,OUTROS,"8,157,440.00","8,112,325.00",1.01
Salto - SP,São Paulo,OUTROS,"7,251,925.00","4,392,135.00",1.65
Avaré - SP,São Paulo,OUTROS,"7,052,748.00","6,687,200.00",1.05
Campo Largo - PR,Paraná,OUTROS,"6,318,676.00","4,177,405.00",1.51


In [46]:
# ============================================================
# 7) GERAR EXCEL (RESUMO + DETALHES + TOP MUNICIPIOS)
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from datetime import datetime

VE = "1A4731"
VM = "2D6A4F"
CZ = "F5F5F5"

def est(ws, r, c, v=None, bold=False, bg=None, fg="222222", aln="left", fmt=None, ind=0):
    cell = ws.cell(row=r, column=c)
    if v is not None:
        cell.value = v
    cell.font = Font(name="Arial", bold=bold, color=fg, size=10)
    cell.alignment = Alignment(horizontal=aln, vertical="center", indent=ind)
    if bg:
        cell.fill = PatternFill("solid", fgColor=bg)
    if fmt:
        cell.number_format = fmt
    return cell

def ban(ws, row, c1, c2, txt, bg=VE, fg="FFFFFF", sz=11, h=24):
    ws.merge_cells(start_row=row, start_column=c1, end_row=row, end_column=c2)
    c = ws.cell(row=row, column=c1)
    c.value = txt
    c.font = Font(name="Arial", bold=True, color=fg, size=sz)
    c.fill = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[row].height = h

def cab(ws, row, cols, bg=VE):
    for j, t in enumerate(cols, 2):
        c = ws.cell(row=row, column=j, value=t)
        c.font = Font(name="Arial", bold=True, color="FFFFFF", size=9)
        c.fill = PatternFill("solid", fgColor=bg)
        c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[row].height = 18

wb = Workbook()

# Resumo
ws_r = wb.active
ws_r.title = "Resumo"
ws_r.sheet_view.showGridLines = False
for col, w in zip("ABCDEFGHI", [2, 8, 24, 12, 16, 16, 12, 12, 2]):
    ws_r.column_dimensions[col].width = w

ban(ws_r, 1, 2, 8, "PINE CHEMICALS - EXPORTACOES POR MUNICIPIO | ComexStat MDIC")
ban(ws_r, 2, 2, 8, f"Periodo: {ANO_INICIAL}-{ANO_FINAL} | {datetime.now().strftime('%d/%m/%Y %H:%M')}", bg=VM, sz=9, h=18)
cab(ws_r, 4, ["SH4", "Produto", "Registros", "FOB Total (USD)", "KG Total", "USD/kg", "Municipios"])

lr = 5
for sh4, desc in SH4S.items():
    d = todos_dados.get(sh4, [])
    bg = CZ if lr % 2 == 0 else "FFFFFF"
    if d:
        df_r = pd.DataFrame(d)
        fob_t = float(df_r["fob_usd"].sum())
        kg_t  = float(df_r["kg_liquido"].sum())
        n_m   = int(df_r["municipio"].nunique())
        n_r   = int(len(df_r))
        ukg   = fob_t / kg_t if kg_t > 0 else 0.0
    else:
        fob_t = kg_t = ukg = 0.0
        n_m = n_r = 0

    est(ws_r, lr, 2, sh4, bold=True, bg=bg, aln="center")
    est(ws_r, lr, 3, desc, bg=bg, ind=1)
    est(ws_r, lr, 4, n_r, bg=bg, aln="right", fmt="#,##0")
    est(ws_r, lr, 5, fob_t, bg=bg, aln="right", fmt="#,##0.00")
    est(ws_r, lr, 6, kg_t, bg=bg, aln="right", fmt="#,##0")
    est(ws_r, lr, 7, round(ukg, 4), bg=bg, aln="right", fmt="0.0000")
    est(ws_r, lr, 8, n_m, bg=bg, aln="center")
    ws_r.row_dimensions[lr].height = 17
    lr += 1

# Abas detalhadas por SH4
for sh4, desc in SH4S.items():
    dados = todos_dados.get(sh4, [])
    ws_d = wb.create_sheet(f"D_{sh4}")
    ws_d.sheet_view.showGridLines = False
    for col, w in zip("ABCDEFGHIJKL", [2, 8, 6, 32, 6, 28, 24, 8, 16, 14, 12, 2]):
        ws_d.column_dimensions[col].width = w

    ban(ws_d, 1, 2, 10, f"{desc}  SH4 {sh4} | {len(dados)} registros")
    ban(ws_d, 2, 2, 10, f"Fluxo: {FLUXO.upper()} | {ANO_INICIAL}-{ANO_FINAL}", bg=VM, sz=9, h=18)
    cab(ws_d, 4, ["Ano", "Mes", "Municipio", "UF", "Pais Destino", "Fabricante", "SH4", "FOB (USD)", "KG Liq", "USD/kg"])

    for i, reg in enumerate(dados, 5):
        bg = CZ if i % 2 == 0 else "FFFFFF"
        vals = [
            reg["ano"], reg["mes"], reg["municipio"], reg["uf"],
            reg.get("pais", "N/D"), reg["fabricante"], reg["sh4"],
            reg["fob_usd"], reg["kg_liquido"], reg["preco_usd_kg"]
        ]
        fmts = [None, None, None, None, None, None, None, "#,##0.00", "#,##0", "0.0000"]
        alns = ["center", "center", "left", "center", "left", "left", "center", "right", "right", "right"]
        for j, (v, f, a) in enumerate(zip(vals, fmts, alns), 2):
            est(ws_d, i, j, v, bg=bg, aln=a, fmt=f)
        ws_d.row_dimensions[i].height = 16

# Top Municipios consolidado (com fabricante)
ws_t = wb.create_sheet("Top_Municipios")
ws_t.sheet_view.showGridLines = False
for col, w in zip("ABCDEFGHIJ", [2, 32, 6, 24, 8, 20, 18, 14, 12, 2]):
    ws_t.column_dimensions[col].width = w

ban(ws_t, 1, 2, 9, "TOP 50 MUNICIPIOS - Consolidado Pine Chemicals")
ban(ws_t, 2, 2, 9, "Ranking por FOB total | Todos SH4 somados", bg=VM, sz=9, h=18)
cab(ws_t, 4, ["Municipio", "UF", "Fabricante", "SH4", "Produto", "FOB (USD)", "KG", "Rank"])

all_df = []
for sh4, desc in SH4S.items():
    d = todos_dados.get(sh4, [])
    if d:
        df_t = pd.DataFrame(d)
        df_t["sh4_c"] = sh4
        df_t["sh4_p"] = desc
        all_df.append(df_t)

if all_df:
    df_all = pd.concat(all_df, ignore_index=True)
    top = (
        df_all.groupby(["municipio", "uf", "fabricante", "sh4_c", "sh4_p"])
        .agg(fob=("fob_usd", "sum"), kg=("kg_liquido", "sum"))
        .reset_index()
        .sort_values("fob", ascending=False)
        .head(50)
    )
    top["rank"] = range(1, len(top) + 1)

    for i, row in enumerate(top.itertuples(index=False), 5):
        bg = CZ if i % 2 == 0 else "FFFFFF"
        est(ws_t, i, 2, row.municipio, bg=bg, aln="left", ind=1)
        est(ws_t, i, 3, row.uf, bg=bg, aln="center")
        est(ws_t, i, 4, row.fabricante, bg=bg, aln="left")
        est(ws_t, i, 5, row.sh4_c, bg=bg, aln="center", bold=True)
        est(ws_t, i, 6, row.sh4_p, bg=bg, aln="left")
        est(ws_t, i, 7, row.fob, bg=bg, aln="right", fmt="#,##0.00")
        est(ws_t, i, 8, row.kg, bg=bg, aln="right", fmt="#,##0")
        est(ws_t, i, 9, row.rank, bg=bg, aln="center")
        ws_t.row_dimensions[i].height = 16

NOME = f"comexstat_municipios_pine_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
wb.save(NOME)
print(f"Excel gerado: {NOME}")


Excel gerado: comexstat_municipios_pine_20260302_1923.xlsx


In [ ]:
# ============================================================
# 8) DOWNLOAD (somente no Google Colab)
# ============================================================
from google.colab import files
files.download(NOME)